# Part 2 — Running the Agentic Claims System in SageMaker Studio

This notebook continues from **Part 1 (AWS-Integration-Addendum.md)**. Instead of running the agents as local Python scripts, we run them as notebook cells inside SageMaker Studio.

**Why this works well here:** every SageMaker Studio notebook runs under an **execution role** (visible under Studio > your user profile > Execution role). As long as that role has the two policies below attached, you don't need `aws configure` or a local `.env` file at all — `boto3` picks up credentials automatically.

Attach these to your SageMaker execution role before running this notebook:
- `secretsmanager:GetSecretValue` on `arn:aws:secretsmanager:*:*:secret:insurance-claims/anthropic-api-key-*`
- `s3:GetObject`, `s3:PutObject`, `s3:ListBucket` on your `insurance-claims-history-<suffix>` bucket

**Prerequisite:** you've already run the `aws secretsmanager create-secret` and `aws s3 mb` commands from Part 1 (from CloudShell, your laptop, or a terminal tab inside Studio itself — Studio has a terminal too).

## 1. Install dependencies
SageMaker Studio's default kernel doesn't have `anthropic` pre-installed.

In [1]:
%pip install -q anthropic boto3

Note: you may need to restart the kernel to use updated packages.


## 1b. Create the secret (run once, safe to re-run)
This checks if `insurance-claims/anthropic-api-key` already exists in Secrets Manager. If it doesn't, it prompts you for your Anthropic API key (via `getpass`, so it's never printed or saved in the notebook file) and creates it.

Get your key from https://console.anthropic.com if you don't have one handy.

In [1]:
import boto3
import json
from getpass import getpass

SECRET_NAME = "insurance-claims/anthropic-api-key"
REGION = boto3.session.Session().region_name or "us-east-1"
secrets_client = boto3.client("secretsmanager", region_name=REGION)

new_key = getpass("Paste your Anthropic API key (starts with sk-ant-): ")

# sanity check BEFORE saving, so we don't repeat the same mistake
assert new_key.startswith("sk-ant-"), f"This doesn't look like a valid key — it starts with '{new_key[:15]}'. Did you paste the right thing?"

response = secrets_client.put_secret_value(
    SecretId=SECRET_NAME,
    SecretString=json.dumps({"ANTHROPIC_API_KEY": new_key}),
)
print("put_secret_value succeeded. VersionId:", response["VersionId"])
del new_key

put_secret_value succeeded. VersionId: dc32e9c4-b323-40df-850d-5d19c7721360


## 2. Load the Claude API key from Secrets Manager
Same secret you created in Part 1 — no key is ever pasted into this notebook.

In [2]:
def get_anthropic_key(secret_name=SECRET_NAME, region=REGION):
    client = boto3.client("secretsmanager", region_name=region)
    response = client.get_secret_value(SecretId=secret_name)
    secret = json.loads(response["SecretString"])
    return secret["ANTHROPIC_API_KEY"]

ANTHROPIC_API_KEY = get_anthropic_key()
print("Key loaded:", bool(ANTHROPIC_API_KEY))

Key loaded: True


## 3. Claude client + a minimal claim-analysis agent
This mirrors `01_local_demo/basic_agent.py` from the original lab, just adapted to run as a notebook cell.

In [3]:
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
MODEL = "claude-sonnet-4-6"

def analyze_claim(claim_text):
    response = client.messages.create(
        model=MODEL,
        max_tokens=500,
        messages=[
            {
                "role": "user",
                "content": (
                    "You are an insurance claims triage agent. "
                    "Given the claim below, return a decision (approve/deny/escalate), "
                    "a risk level (low/medium/high), and a one-sentence reason.\n\n"
                    f"Claim: {claim_text}"
                ),
            }
        ],
    )
    return response.content[0].text

sample_claim = "Rear-end collision, $3,200 in vehicle damage, no injuries, policy active for 2 years, no prior claims."
result = analyze_claim(sample_claim)
print(result)

## Claims Triage Decision

| Field | Decision |
|-------|----------|
| **Decision** | ✅ Approve |
| **Risk Level** | 🟢 Low |
| **Reason** | Straightforward rear-end collision with moderate property damage, no injuries, an active policy in good standing, and a clean two-year claims history presents minimal fraud or liability risk. |


## 3b. Create the S3 bucket (if it doesn't exist yet)
Run this once. It's safe to re-run — if the bucket already exists and you own it, it just skips creation.

We derive a unique bucket name from your AWS account ID so you don't have to hand-pick a globally-unique name.

In [4]:
import botocore

sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]
BUCKET = f"insurance-claims-history-{account_id}"

s3_client = boto3.client("s3", region_name=REGION)

try:
    if REGION == "us-east-1":
        s3_client.create_bucket(Bucket=BUCKET)
    else:
        s3_client.create_bucket(
            Bucket=BUCKET,
            CreateBucketConfiguration={"LocationConstraint": REGION},
        )
    print(f"Created bucket: {BUCKET}")
except s3_client.exceptions.BucketAlreadyOwnedByYou:
    print(f"Bucket already exists and is owned by you: {BUCKET}")
except botocore.exceptions.ClientError as e:
    if e.response["Error"]["Code"] == "BucketAlreadyExists":
        raise RuntimeError(
            f"Bucket name '{BUCKET}' is taken by someone else globally — "
            "this shouldn't happen with the account-id suffix, but if it does, "
            "add a manual suffix, e.g. BUCKET = f'{BUCKET}-lab1'"
        )
    raise

print("Using bucket:", BUCKET)

Created bucket: insurance-claims-history-442042521384
Using bucket: insurance-claims-history-442042521384


## 4. Persist the result to S3 (Stateful Agent, S3-backed)
Same `S3ClaimMemory` pattern from Part 1's `03_cloud_integration/s3_memory.py` — reused here directly so history survives across notebook restarts.

In [5]:
from datetime import datetime, timezone


class S3ClaimMemory:
    def __init__(self, bucket, prefix="claims/"):
        self.s3 = boto3.client("s3")
        self.bucket = bucket
        self.prefix = prefix

    def save_claim(self, claim_id, record):
        key = f"{self.prefix}{claim_id}.json"
        record["saved_at"] = datetime.now(timezone.utc).isoformat()
        self.s3.put_object(Bucket=self.bucket, Key=key, Body=json.dumps(record).encode("utf-8"))
        return key

    def load_claim(self, claim_id):
        key = f"{self.prefix}{claim_id}.json"
        obj = self.s3.get_object(Bucket=self.bucket, Key=key)
        return json.loads(obj["Body"].read())

    def list_claims(self):
        paginator = self.s3.get_paginator("list_objects_v2")
        keys = []
        for page in paginator.paginate(Bucket=self.bucket, Prefix=self.prefix):
            for obj in page.get("Contents", []):
                keys.append(obj["Key"])
        return keys

memory = S3ClaimMemory(BUCKET)
saved_key = memory.save_claim("claim_001", {"claim_text": sample_claim, "analysis": result})
print("Saved to:", saved_key)

Saved to: claims/claim_001.json


In [6]:
# Verify it round-trips
loaded = memory.load_claim("claim_001")
print(json.dumps(loaded, indent=2))

{
  "claim_text": "Rear-end collision, $3,200 in vehicle damage, no injuries, policy active for 2 years, no prior claims.",
  "analysis": "## Claims Triage Decision\n\n| Field | Decision |\n|-------|----------|\n| **Decision** | \u2705 Approve |\n| **Risk Level** | \ud83d\udfe2 Low |\n| **Reason** | Straightforward rear-end collision with moderate property damage, no injuries, an active policy in good standing, and a clean two-year claims history presents minimal fraud or liability risk. |",
  "saved_at": "2026-08-08T07:36:03.675209+00:00"
}


## 5. Batch-process multiple claims and list history
This is roughly the SageMaker-native equivalent of `full_system.py` — instead of a CLI script, you get an inspectable, re-runnable notebook you can iterate on cell by cell.

In [7]:
claims_batch = [
    ("claim_002", "Water damage from burst pipe, $12,000 estimated repair, policy 6 months old."),
    ("claim_003", "Minor fender bender, $800 damage, 5-year policy holder, 3 prior claims in 2 years."),
]

for claim_id, text in claims_batch:
    analysis = analyze_claim(text)
    memory.save_claim(claim_id, {"claim_text": text, "analysis": analysis})
    print(f"--- {claim_id} ---")
    print(analysis)
    print()

print("All claims in S3:", memory.list_claims())

--- claim_002 ---
## Claims Triage Decision

| Field | Assessment |
|-------|------------|
| **Decision** | **Approve** |
| **Risk Level** | **Low** |
| **Reason** | Burst pipe water damage is a standard covered peril, the claim amount is within typical policy limits, and the 6-month-old active policy shows no indicators of pre-existing conditions or fraud risk. |



--- claim_003 ---
## Claims Triage Decision

| Field | Value |
|-------|-------|
| **Decision** | Escalate |
| **Risk Level** | Medium |
| **Reason** | While the damage amount is low and the policyholder has a long tenure, the elevated claim frequency (3 claims in 2 years) warrants further review before approval to assess potential pattern of abuse or high-risk behavior. |

All claims in S3: ['claims/claim_001.json', 'claims/claim_002.json', 'claims/claim_003.json', 'claims/claim_004.json']


## 6. Router Agent
Classifies the claim (type + complexity) and picks which handler it should go to. In the full pipeline this decides which of the other patterns run next.

In [8]:
import re

def extract_json(text):
    """Claude sometimes wraps JSON responses in ```json ... ``` fences -- strip them before parsing."""
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if match:
        text = match.group(1)
    return json.loads(text.strip())

def route_claim(claim_text):
    response = client.messages.create(
        model=MODEL,
        max_tokens=200,
        messages=[{
            "role": "user",
            "content": (
                "Classify this insurance claim. Return ONLY valid JSON, no markdown code fences, "
                "no prose, no explanation — just the raw JSON object with keys: "
                "claim_type (auto/property/health/other), complexity (simple/complex), "
                "route (fast_track/standard_review/specialist_review).\n\n"
                f"Claim: {claim_text}"
            ),
        }],
    )
    return extract_json(response.content[0].text)

## 7. Guardrail Agent
Checks a proposed decision against business/compliance rules before it's allowed to finalize.

In [9]:
GUARDRAIL_RULES = [
    "Claims over $50,000 must be escalated, never auto-approved.",
    "Claims with no supporting details (damage amount, cause) must be escalated for more info.",
    "Policies active less than 30 days require manual review regardless of claim size.",
]

def guardrail_check(claim_text, proposed_decision):
    rules_text = "\n".join(f"- {r}" for r in GUARDRAIL_RULES)
    response = client.messages.create(
        model=MODEL,
        max_tokens=300,
        messages=[{
            "role": "user",
            "content": (
                f"Rules:\n{rules_text}\n\n"
                f"Claim: {claim_text}\n"
                f"Proposed decision: {proposed_decision}\n\n"
                "Does the proposed decision violate any rule? Answer with PASS or FAIL "
                "followed by a one-sentence reason."
            ),
        }],
    )
    return response.content[0].text

print(guardrail_check(sample_claim, "approve"))

PASS — The claim is under $50,000, includes sufficient supporting details (damage amount and cause), and the policy has been active for well over 30 days, so none of the escalation or manual review rules are triggered.


## 8. RAG Agent (policy-grounded)
Retrieves a relevant policy snippet before generating a decision, instead of relying on the model's assumptions. Here the "knowledge base" is a small in-notebook dict — swap this for a real vector store (e.g. OpenSearch Serverless or a Bedrock Knowledge Base) for production.

In [10]:
POLICY_KB = {
    "auto": "Auto policy covers collision damage up to $25,000 per incident, with a $500 deductible. "
            "Claims must be filed within 30 days of the incident.",
    "property": "Property policy covers water damage from sudden pipe failure, excluding gradual leaks or "
                "neglect. Coverage cap is $75,000 per incident.",
}

def rag_decision(claim_text, claim_type):
    policy_snippet = POLICY_KB.get(claim_type, "No specific policy on file; use general underwriting guidance.")
    response = client.messages.create(
        model=MODEL,
        max_tokens=300,
        messages=[{
            "role": "user",
            "content": (
                f"Relevant policy text: {policy_snippet}\n\n"
                f"Claim: {claim_text}\n\n"
                "Using only the policy text above, decide approve/deny/escalate and explain why, "
                "citing the specific policy terms that apply."
            ),
        }],
    )
    return response.content[0].text

print(rag_decision(sample_claim, "auto"))

## Claim Decision: **APPROVE**

---

### Reasoning:

Evaluating the claim against each applicable policy term:

**1. Coverage Type — Met**
The claim involves a rear-end collision, which constitutes collision damage. The policy explicitly covers collision damage.

**2. Coverage Limit — Met**
The damage amount is **$3,200**, which is well within the **$25,000 per incident maximum**. No coverage gap exists here.

**3. Deductible — Applied**
Per policy terms, a **$500 deductible applies**. The approved payout should therefore be:
> $3,200 − $500 = **$2,700 payable to claimant**

**4. Filing Window — Cannot Fully Confirm**
The policy requires claims to be filed **within 30 days of the incident**. The provided claim information **does not specify the filing date or incident date**. This should be verified before final disbursement, but does not alone warrant escalation if standard intake procedures captured this.

---

### Notes / Caveats:
- Policy tenure (2 years) and claims history are **n

## 9. Tool Agent
Calls an external function as part of its reasoning, rather than relying purely on generated text. Example: a repair-cost lookup "tool".

In [12]:
def serialize_content(content):
    """Convert SDK response objects to plain dicts for the messages list."""
    result = []
    for block in content:
        if block.type == "text":
            result.append({"type": "text", "text": block.text})
        elif block.type == "tool_use":
            result.append({
                "type": "tool_use",
                "id": block.id,
                "name": block.name,
                "input": block.input
            })
    return result

def tool_agent(claim_text):
    messages = [{"role": "user", "content": f"Claim: {claim_text}\n\nLook up the average repair cost for the relevant damage type, then assess if the claimed amount is reasonable."}]
    response = client.messages.create(model=MODEL, max_tokens=500, tools=tools, messages=messages)

    if response.stop_reason == "tool_use":
        tool_call = next(b for b in response.content if b.type == "tool_use")
        tool_result = lookup_average_repair_cost(**tool_call.input)

        # ← key fix: serialize to plain dicts before appending
        messages.append({"role": "assistant", "content": serialize_content(response.content)})
        messages.append({
            "role": "user",
            "content": [{"type": "tool_result", "tool_use_id": tool_call.id, "content": tool_result}],
        })
        final = client.messages.create(model=MODEL, max_tokens=400, tools=tools, messages=messages)
        return final.content[0].text
    return response.content[0].text

print(tool_agent(sample_claim))

Here's a full assessment of the claim based on the lookup results:

---

## 🚗 Claim Assessment Report

### 📋 Claim Summary
| Field | Details |
|---|---|
| **Damage Type** | Rear-end collision |
| **Claimed Amount** | $3,200 |
| **Injuries** | None |
| **Policy Status** | Active (2 years) |
| **Prior Claims** | None |

---

### 💰 Cost Reasonableness Analysis
| Metric | Value |
|---|---|
| **Average Repair Cost** (Rear-End Collision) | $5,000 |
| **Claimed Amount** | $3,200 |
| **Variance** | -$1,800 (36% below average) |

✅ **The claimed amount of $3,200 is reasonable.** It falls **well below** the industry average of $5,000 for rear-end collision repairs, suggesting no inflation or exaggeration of costs.

---

### 🔍 Additional Risk Indicators
| Factor | Assessment |
|---|---|
| **Claim Amount vs. Average** | ✅ Low Risk — 36% below average |
| **Injuries Reported** | ✅ Low Risk — No injuries, reducing liability concerns |
| **Policy Tenure** | ✅ Low Risk — 2-year active policy indicates

## 10. Reflection Agent
Reviews and critiques the system's own prior output before it's returned as final.

In [13]:
def reflect_on_decision(claim_text, initial_decision):
    response = client.messages.create(
        model=MODEL,
        max_tokens=400,
        messages=[{
            "role": "user",
            "content": (
                f"Claim: {claim_text}\n"
                f"Initial decision: {initial_decision}\n\n"
                "Critique this decision — is anything missing, inconsistent, or unjustified? "
                "Then give an improved final decision."
            ),
        }],
    )
    return response.content[0].text

print(reflect_on_decision(sample_claim, result))

## Critical Review of Initial Decision

### What the Initial Decision Gets Right
- Recognizing clean claims history as a positive factor
- Noting active policy status
- Correctly identifying absence of injuries as a risk reducer
- Reasonable approval direction overall

---

### Problems and Gaps

**1. Missing Information — Significant Underspecification**
The claim provides almost no actionable detail:
- **Who was at fault?** Rear-end doesn't automatically mean the other party is liable in all jurisdictions or circumstances
- **Was a police report filed?** Absence is a meaningful fraud indicator
- **Were photos/repair estimates submitted?** $3,200 figure — is this claimed or appraised?
- **What coverage type applies?** Collision vs. liability vs. uninsured motorist matters substantially
- **Identity of other driver** and whether *their* insurer was notified
- **Date of incident vs. date of claim** — delay is a flag
- **Vehicle details** — $3,200 on a high-value car vs. a low-value car 

## 11. Human Escalation Agent
Flags claims that shouldn't be auto-resolved and need a person to look at them.

In [14]:
def needs_human_escalation(claim_text):
    response = client.messages.create(
        model=MODEL,
        max_tokens=200,
        messages=[{
            "role": "user",
            "content": (
                f"Claim: {claim_text}\n\n"
                "Should this claim be escalated to a human reviewer? Consider: high dollar amount, "
                "ambiguous cause, missing details, or fraud indicators. "
                "Answer ESCALATE or AUTO_PROCESS with a one-sentence reason."
            ),
        }],
    )
    return response.content[0].text

print(needs_human_escalation(sample_claim))

**AUTO_PROCESS**

This claim presents a straightforward rear-end collision with a moderate, believable damage amount, no injuries, a stable two-year policy history, no prior claims, and no ambiguous cause or fraud indicators requiring human review.


## 12. Planner Agent
Breaks the claim down into a step-by-step processing plan up front, instead of resolving everything in one shot.

In [15]:
def plan_claim_processing(claim_text):
    response = client.messages.create(
        model=MODEL,
        max_tokens=400,
        messages=[{
            "role": "user",
            "content": (
                f"Claim: {claim_text}\n\n"
                "Produce a numbered, step-by-step plan (3-6 steps) for how this claim should be "
                "processed — e.g. triage, verification, policy check, decision, notification."
            ),
        }],
    )
    return response.content[0].text

print(plan_claim_processing(sample_claim))

# Claim Processing Plan: Rear-End Collision ($3,200 Vehicle Damage)

---

## Step 1 — Intake & Triage
- Assign a unique claim ID and timestamp the submission
- Classify severity: **low complexity** (property damage only, no injuries, clear-cut fault scenario)
- Route to standard auto claims queue; no urgent escalation required
- Confirm claimant contact information is complete

---

## Step 2 — Policy Verification
- Confirm policy is **active** and coverage was in force on the date of loss
- Verify collision or comprehensive coverage applies to this incident type
- Pull **deductible amount** and coverage limits from the policy record
- Note: 2-year tenure, zero prior claims — flags as **low-risk, preferred customer**

---

## Step 3 — Incident & Liability Verification
- Collect police report, photos, and any available dashcam or witness statements
- Confirm rear-end fault assignment (typically straightforward)
- Determine whether subrogation against the at-fault third party is applicab

## 13. Full pipeline — all 8 patterns chained
This is the SageMaker-notebook equivalent of `full_system.py`: Plan → Route → RAG-grounded decision → Tool check → Guardrail → Reflect → Escalation check → save to S3 (Stateful).

In [20]:
import json

def tool_agent(claim_text):
    messages = [{
        "role": "user",
        "content": f"Claim: {claim_text}\n\nLook up the average repair cost for the relevant damage type, then assess if the claimed amount is reasonable."
    }]

    response = client.messages.create(model=MODEL, max_tokens=500, tools=tools, messages=messages)

    if response.stop_reason == "tool_use":
        # exclude_none=True strips null fields that confuse the API
        content = json.loads(response.model_dump_json(exclude_none=True))["content"]

        tool_call = next(b for b in content if b["type"] == "tool_use")
        tool_result = lookup_average_repair_cost(**tool_call["input"])

        messages.append({"role": "assistant", "content": content})
        messages.append({
            "role": "user",
            "content": [{
                "type": "tool_result",
                "tool_use_id": tool_call["id"],
                "content": tool_result
            }]
        })

        final = client.messages.create(model=MODEL, max_tokens=400, tools=tools, messages=messages)
        return next((b.text for b in final.content if b.type == "text"), "Tool analysis complete.")

    return next((b.text for b in response.content if b.type == "text"), "No result.")

print(tool_agent(sample_claim))

Here's a full assessment of the claim based on the data retrieved:

---

## 🚗 Claim Assessment Report

### 📋 Claim Summary
| Field | Detail |
|---|---|
| **Damage Type** | Rear-end collision |
| **Claimed Amount** | $3,200 |
| **Injuries** | None |
| **Policy Active** | 2 Years |
| **Prior Claims** | None |

---

### 💰 Cost Reasonableness Check
| Metric | Value |
|---|---|
| **Average Repair Cost (Rear-End Collision)** | $5,000 |
| **Claimed Amount** | $3,200 |
| **Variance from Average** | -$1,800 (-36%) |

✅ **The claimed amount of $3,200 is reasonable and below the industry average of $5,000** for rear-end collision repairs. This reduces the likelihood of cost inflation or fraud.

---

### 🔍 Overall Risk Indicators
| Factor | Assessment |
|---|---|
| Claimed amount vs. average | ✅ Low risk — 36% below average |
| Injuries reported | ✅ Low risk — No injuries |
| Policy tenure | ✅ Low risk — Active for 2 years (established customer) |
| Prior claims history | ✅ Low risk — No prior cla

In [21]:
full_result = run_full_pipeline("claim_004", sample_claim)
print(json.dumps(full_result, indent=2))

{
  "claim_text": "Rear-end collision, $3,200 in vehicle damage, no injuries, policy active for 2 years, no prior claims.",
  "plan": "# Claim Processing Plan: Rear-End Collision ($3,200 Vehicle Damage)\n\n---\n\n## Step 1: Intake & Triage\n- Log claim into the system with timestamp and assign a unique claim ID\n- Classify severity: **Low complexity** (property damage only, no injuries, clear liability scenario)\n- Assign to a standard claims adjuster \u2014 no specialist required\n- Confirm all required documentation is submitted (photos, police report if available, repair estimate)\n\n---\n\n## Step 2: Policy Verification\n- Confirm policy is **active and in good standing** at the date of loss \u2713\n- Verify coverage type includes **collision or property damage liability**\n- Identify applicable **deductible amount** (to be subtracted from payout)\n- Note **2-year tenure and zero prior claims** \u2014 flags account as low-risk\n\n---\n\n## Step 3: Liability & Damage Assessment\n- C

---
# Part 3 — Deploying the app from inside SageMaker Studio

Studio's kernel image doesn't have a Docker daemon, so we can't `docker build` directly in a notebook cell. Instead we use **`sagemaker-studio-image-build`**, which sends the build to AWS CodeBuild and pushes the resulting image to ECR — all triggered from a single CLI command run inside Studio.

**One-time setup needed on your SageMaker execution role** (or a role CodeBuild assumes):
- `AmazonEC2ContainerRegistryFullAccess` (or scoped ECR push/pull)
- `AWSCodeBuildAdminAccess` (or scoped CodeBuild start/read)
- Trust relationship allowing `codebuild.amazonaws.com` to assume the role

See: https://github.com/aws-samples/sagemaker-studio-image-build-cli for the exact IAM policy JSON.

In [22]:
%pip install -q sagemaker-studio-image-build

Note: you may need to restart the kernel to use updated packages.


### Write the Streamlit app, requirements, and Dockerfile
This packages the same agent functions defined above into a minimal web UI.

In [23]:
%%writefile app.py
import streamlit as st
import boto3, json, os
import anthropic

def get_anthropic_key(secret_name="insurance-claims/anthropic-api-key", region="us-east-1"):
    client = boto3.client("secretsmanager", region_name=region)
    response = client.get_secret_value(SecretId=secret_name)
    return json.loads(response["SecretString"])["ANTHROPIC_API_KEY"]

client = anthropic.Anthropic(api_key=get_anthropic_key())
MODEL = "claude-sonnet-4-6"

st.title("Insurance Claim Processing System")
claim_text = st.text_area("Claim details")

if st.button("Analyze Claim") and claim_text:
    with st.spinner("Running agentic pipeline..."):
        response = client.messages.create(
            model=MODEL,
            max_tokens=500,
            messages=[{"role": "user", "content": f"Analyze this insurance claim and give a decision (approve/deny/escalate), risk level, and reasoning: {claim_text}"}],
        )
        st.write(response.content[0].text)

Overwriting app.py


In [24]:
%%writefile requirements.txt
streamlit
anthropic
boto3

Overwriting requirements.txt


In [25]:
%%writefile Dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8501
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0"]

Overwriting Dockerfile


### Build and push to ECR (runs via CodeBuild, no local Docker needed)

In [26]:
!sm-docker build . --repository insurance-claims-ui:latest

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()


.

.

.

...


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Removing intermediate container b58928b2fe14
 ---> b1df7afaf370
Step 5/7 : COPY . .
 ---> db6493c9af18
Step 6/7 : EXPOSE 8501
 ---> Running in 2da628e80ce3
Removing intermediate container 2da628e80ce3
 ---> 0a448fdc9e0f
Step 7/7 : CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0"]
 ---> Running in b7e22127e806
Removing intermediate container b7e22127e806
 ---> f18cd3ea63dc
Successfully built f18cd3ea63dc
Successfully tagged insurance-claims-ui:latest

[Container] 2026/08/08 07:52:10.617112 Running command docker tag $IMAGE_REPO_NAME:$IMAGE_TAG $AWS_ACCOUNT_ID.dkr.ecr.$AWS_DEFAULT_REGION.amazonaws.com/$IMAGE_REPO_NAME:$IMAGE_TAG

[Container] 2026/08/08 07:52:10.642437 Phase complete: BUILD State: SUCCEEDED
[Container] 2026/08/08 07:52:10.642454 Phase context status code:  Message:
[Container] 2026/08/08 07:52:10.682533 Entering phase POST_BUILD
[Container] 2026/08/08 07:52:10.683667 Running command echo Build completed on `date`
Build completed on Sat A

6f9432833129: Pushed


42d53538bf46: Pushed
latest: digest: sha256:1f0023755aeb6bff86f205731289a01358179a8a43f37e036257124ca8b743e9 size: 1997

[Container] 2026/08/08 07:52:32.064945 Phase complete: POST_BUILD State: SUCCEEDED
[Container] 2026/08/08 07:52:32.064963 Phase context status code:  Message:
[Container] 2026/08/08 07:52:32.109643 Set report auto-discover timeout to 5 seconds
[Container] 2026/08/08 07:52:32.109684 Expanding base directory path:  .
[Container] 2026/08/08 07:52:32.111035 Assembling file list
[Container] 2026/08/08 07:52:32.111048 Expanding .
[Container] 2026/08/08 07:52:32.112454 Expanding file paths for base directory .
[Container] 2026/08/08 07:52:32.112465 Assembling file list
[Container] 2026/08/08 07:52:32.112468 Expanding **/*
[Container] 2026/08/08 07:52:32.119402 No matching auto-discover report paths found
[Container] 2026/08/08 07:52:32.119424 Report auto-discover file discovery took 0.009780 seconds
[Container] 2026/08/08 07:52:32.119526 Phase complete: UPLOAD_ARTIFACTS Sta

Image URI: 442042521384.dkr.ecr.us-east-1.amazonaws.com/insurance-claims-ui:latest


Copy the **image URI** printed at the end of that command's output — you'll need it for the next step.

### Updated app.py — lazy key loading
The Anthropic client is created inside `@st.cache_resource` so the container does not crash at startup.

In [27]:
%%writefile app.py
import streamlit as st
import boto3, json
import anthropic

@st.cache_resource
def get_client():
    def _get_key(secret_name="insurance-claims/anthropic-api-key", region="us-east-1"):
        sm = boto3.client("secretsmanager", region_name=region)
        resp = sm.get_secret_value(SecretId=secret_name)
        return json.loads(resp["SecretString"])["ANTHROPIC_API_KEY"]
    return anthropic.Anthropic(api_key=_get_key())

MODEL = "claude-sonnet-4-6"

st.title("Insurance Claim Processing System")
claim_text = st.text_area("Claim details")

if st.button("Analyze Claim") and claim_text:
    with st.spinner("Running agentic pipeline..."):
        client = get_client()
        response = client.messages.create(
            model=MODEL,
            max_tokens=500,
            messages=[{"role": "user", "content": f"Analyze this insurance claim and give a decision (approve/deny/escalate), risk level, and reasoning: {claim_text}"}],
        )
        st.write(response.content[0].text)


Overwriting app.py


---
# Part 3 — Deploy to EC2

> **Note:** AWS App Runner stopped accepting new customers on April 30, 2026.
> EC2 is the recommended replacement for this lab.

The ECR image built in Part 2 is reused as-is — no rebuild needed.
We launch a `t3.small` Amazon Linux 2 instance, install Docker, pull the image, and run Streamlit on port 8501.

#### Step 1 — Create the EC2 IAM role and instance profile (run once)

In [30]:
import boto3, json, time

iam = boto3.client("iam")
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
REGION = boto3.session.Session().region_name or "us-east-1"
ROLE_NAME = "EC2InsuranceClaimsRole"

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "ec2.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

try:
    iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="EC2 role for insurance-claims Streamlit app"
    )
    print("Role created")
except iam.exceptions.EntityAlreadyExistsException:
    print("Role already exists — skipping")

# ECR read access
iam.attach_role_policy(
    RoleName=ROLE_NAME,
    PolicyArn="arn:aws:iam::aws:policy/AmazonEC2ContainerRegistryReadOnly"
)

# Secrets Manager access
iam.put_role_policy(
    RoleName=ROLE_NAME,
    PolicyName="SecretsManagerAccess",
    PolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": "secretsmanager:GetSecretValue",
            "Resource": f"arn:aws:secretsmanager:{REGION}:{ACCOUNT_ID}:secret:insurance-claims/anthropic-api-key*"
        }]
    })
)

# Instance profile
try:
    iam.create_instance_profile(InstanceProfileName=ROLE_NAME)
    iam.add_role_to_instance_profile(InstanceProfileName=ROLE_NAME, RoleName=ROLE_NAME)
    print("Instance profile created")
except iam.exceptions.EntityAlreadyExistsException:
    print("Instance profile already exists — skipping")

time.sleep(10)  # IAM propagation delay
print("IAM setup complete")


Role created


Instance profile created


IAM setup complete


#### Step 2 — Launch the EC2 instance

In [32]:
ec2 = boto3.client("ec2", region_name=REGION)

user_data = f"""#!/bin/bash
yum update -y
yum install -y docker
systemctl start docker
systemctl enable docker
aws ecr get-login-password --region {REGION} | docker login --username AWS --password-stdin {ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com
docker pull {ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/insurance-claims-ui:latest
docker run -d -p 8501:8501 --restart always {ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/insurance-claims-ui:latest
"""

response = ec2.run_instances(
    ImageId="ami-0c02fb55956c7d316",   # Amazon Linux 2 — us-east-1
    InstanceType="t3.small",
    MinCount=1,
    MaxCount=1,
    UserData=user_data,
    IamInstanceProfile={"Name": ROLE_NAME},
    TagSpecifications=[{
        "ResourceType": "instance",
        "Tags": [{"Key": "Name", "Value": "insurance-claims-ui"}]
    }]
)

instance_id = response["Instances"][0]["InstanceId"]
print("Instance launched:", instance_id)


Instance launched: i-0ba2d3635e5e0c927


#### Step 3 — Open port 8501 in the security group

In [33]:
import time
time.sleep(15)  # wait for instance metadata to populate

instance_info = ec2.describe_instances(InstanceIds=[instance_id])["Reservations"][0]["Instances"][0]
sg_id = instance_info["SecurityGroups"][0]["GroupId"]
print("Security Group:", sg_id)

try:
    ec2.authorize_security_group_ingress(
        GroupId=sg_id,
        IpPermissions=[{
            "IpProtocol": "tcp",
            "FromPort": 8501,
            "ToPort": 8501,
            "IpRanges": [{"CidrIp": "0.0.0.0/0", "Description": "Streamlit"}]
        }]
    )
    print("Port 8501 opened")
except ec2.exceptions.ClientError as e:
    if "already exists" in str(e):
        print("Port 8501 already open")
    else:
        raise


Security Group: sg-0e4f92b5602aca3bf


Port 8501 opened


#### Step 4 — Wait for the instance to be running, then open the app

In [34]:
print("Waiting for instance to reach running state...")
waiter = ec2.get_waiter("instance_running")
waiter.wait(InstanceIds=[instance_id])

instance_info = ec2.describe_instances(InstanceIds=[instance_id])["Reservations"][0]["Instances"][0]
public_ip = instance_info["PublicIpAddress"]

print("Instance running!")
print(f"App will be ready in ~3 minutes (Docker is pulling the image in the background)")
print(f"")
print(f"Open this URL in your browser:")
print(f"  http://{public_ip}:8501")


Waiting for instance to reach running state...


Instance running!
App will be ready in ~3 minutes (Docker is pulling the image in the background)

Open this URL in your browser:
  http://3.236.36.5:8501


#### Step 5 — Terminate when done

> **Cost reminder:** `t3.small` costs ~\$0.02/hour (~\$15/month if left running).
> Run the cell below when you are finished with the demo.


In [42]:
import boto3, json

REGION = boto3.session.Session().region_name or "us-east-1"
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
BUCKET = f"insurance-claims-history-{ACCOUNT_ID}"

ec2 = boto3.client("ec2", region_name=REGION)
ecr = boto3.client("ecr", region_name=REGION)
secrets = boto3.client("secretsmanager", region_name=REGION)
s3 = boto3.resource("s3")
iam = boto3.client("iam")

# 1. Terminate EC2
try:
    ec2.terminate_instances(InstanceIds=[instance_id])
    print("✅ EC2 instance terminated:", instance_id)
except Exception as e:
    print("EC2:", e)

# 2. Delete ECR repository
try:
    ecr.delete_repository(repositoryName="insurance-claims-ui", force=True)
    print("✅ ECR repository deleted")
except Exception as e:
    print("ECR:", e)

# 3. Delete Secrets Manager secret
try:
    secrets.delete_secret(SecretId="insurance-claims/anthropic-api-key", ForceDeleteWithoutRecovery=True)
    print("✅ Secret deleted")
except Exception as e:
    print("Secrets Manager:", e)

# 4. Delete S3 bucket
try:
    bucket = s3.Bucket(BUCKET)
    bucket.objects.all().delete()
    bucket.delete()
    print("✅ S3 bucket deleted:", BUCKET)
except Exception as e:
    print("S3:", e)

# 5. Delete EC2InsuranceClaimsRole
try:
    iam.detach_role_policy(RoleName="EC2InsuranceClaimsRole", PolicyArn="arn:aws:iam::aws:policy/AmazonEC2ContainerRegistryReadOnly")
    iam.delete_role_policy(RoleName="EC2InsuranceClaimsRole", PolicyName="SecretsManagerAccess")
    iam.remove_role_from_instance_profile(InstanceProfileName="EC2InsuranceClaimsRole", RoleName="EC2InsuranceClaimsRole")
    iam.delete_instance_profile(InstanceProfileName="EC2InsuranceClaimsRole")
    iam.delete_role(RoleName="EC2InsuranceClaimsRole")
    print("✅ EC2InsuranceClaimsRole deleted")
except Exception as e:
    print("EC2InsuranceClaimsRole:", e)

# 6. Delete AppRunnerInstanceRole
try:
    iam.delete_role_policy(RoleName="AppRunnerInstanceRole", PolicyName="SecretsManagerAccess")
    iam.delete_role(RoleName="AppRunnerInstanceRole")
    print("✅ AppRunnerInstanceRole deleted")
except Exception as e:
    print("AppRunnerInstanceRole:", e)

print("\n🧹 Cleanup complete!")

✅ EC2 instance terminated: i-0ba2d3635e5e0c927
ECR: An error occurred (AccessDeniedException) when calling the DeleteRepository operation: User: arn:aws:sts::442042521384:assumed-role/AmazonSageMaker-ExecutionRole-20260808T104532/SageMaker is not authorized to perform: ecr:DeleteRepository on resource: arn:aws:ecr:us-east-1:442042521384:repository/insurance-claims-ui because no identity-based policy allows the ecr:DeleteRepository action
✅ Secret deleted


S3: An error occurred (AccessDenied) when calling the DeleteBucket operation: User: arn:aws:sts::442042521384:assumed-role/AmazonSageMaker-ExecutionRole-20260808T104532/SageMaker is not authorized to perform: s3:DeleteBucket on resource: "arn:aws:s3:::insurance-claims-history-442042521384" because no identity-based policy allows the s3:DeleteBucket action


✅ EC2InsuranceClaimsRole deleted
✅ AppRunnerInstanceRole deleted

🧹 Cleanup complete!


## Next steps

- **Turn this into a scheduled job:** SageMaker Studio notebooks can be run headlessly via a **SageMaker Notebook Job** (right-click the notebook > "Create notebook job").
- **Bring in the other 7 patterns:** copy the logic from `02_patterns/*.py` into new cells the same way `analyze_claim()` was adapted above.
- **Scale up:** swap `t3.small` for a larger instance type, or migrate to ECS Fargate for auto-scaling production deployments.
